In [6]:
import torch
import numpy as np
from torch.utils import data

from torch import nn

In [23]:
def synthetic_data(w, b, num_examples):
    # 构造随机数据y = Xw + b + e
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = X @ w + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1,1))

def load_array(data_arrays, batch_size, is_train=True):
    """构造一个PyTorch数据迭代器"""
    # is_train表示是否希望数据迭代器对象在每个迭代周期内打乱数据
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)


In [25]:
true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)

batch_size = 10
data_iter = load_array((features, labels), batch_size)
print(next(iter(data_iter)))

[tensor([[ 0.6717,  0.8559],
        [-0.4414,  0.6232],
        [-0.4051, -0.2540],
        [-2.5442,  2.0475],
        [-1.4893, -1.1498],
        [-0.8275,  2.1141],
        [ 0.0920,  0.5843],
        [-0.0706,  0.0467],
        [ 1.5271, -0.8589],
        [-1.6842,  0.2111]]), tensor([[ 2.6345],
        [ 1.2111],
        [ 4.2490],
        [-7.8560],
        [ 5.1334],
        [-4.6266],
        [ 2.3931],
        [ 3.8940],
        [10.1872],
        [ 0.1216]])]


In [38]:
net = nn.Sequential(nn.Linear(2, 1))
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

loss = nn.MSELoss()
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

num_epochs = 5
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X), y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

w = net[0].weight.data
print('w误差:', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b误差:', true_b - b)

epoch 1, loss 0.000212
epoch 2, loss 0.000098
epoch 3, loss 0.000097
epoch 4, loss 0.000095
epoch 5, loss 0.000096
w误差: tensor([ 0.0006, -0.0003])
b误差: tensor([0.0004])


In [13]:
net(X).flatten().shape


torch.Size([10])